In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split

diabetes = load_diabetes(as_frame=True)   # 판다스 표(DataFrame)로 불러오기
df = diabetes.frame                        # 특성 10개 + target(정답)이 한 표에
X = diabetes.data                          # 특성만 (10열)
y = diabetes.target                        # 정답 (당뇨 진행도)
print('데이터 크기:', df.shape)             # (442, 11) = 442명 x (특성10 + 정답1)
print('결측치:', int(df.isnull().sum().sum()))   # 0 (내장 데이터라 깨끗)
df.head()                                  # 실제 값 미리보기 (표)

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

데이터 크기: (442, 11)
결측치: 0


In [2]:
X

,age,sex,bmi,bp,s1,s2,s3,s4,s5,s6
0,0.038076,0.050680,0.061696,0.021872,-0.044223,-0.034821,-0.043401,-0.002592,0.019907,-0.017646
1,-0.001882,-0.044642,-0.051474,-0.026328,-0.008449,-0.019163,0.074412,-0.039493,-0.068332,-0.092204
2,0.085299,0.050680,0.044451,-0.005670,-0.045599,-0.034194,-0.032356,-0.002592,0.002861,-0.025930
3,-0.089063,-0.044642,-0.011595,-0.036656,0.012191,0.024991,-0.036038,0.034309,0.022688,-0.009362
4,0.005383,-0.044642,-0.036385,0.021872,0.003935,0.015596,0.008142,-0.002592,-0.031988,-0.046641
...,...,...,...,...,...,...,...,...,...,...
437,0.041708,0.050680,0.019662,0.059744,-0.005697,-0.002566,-0.028674,-0.002592,0.031193,0.007207
438,-0.005515,0.050680,-0.015906,-0.067642,0.049341,0.079165,-0.028674,0.034309,-0.018114,0.044485
439,0.041708,0.050680,-0.015906,0.017293,-0.037344,-0.013840,-0.024993,-0.011080,-0.046883,0.015491
440,-0.045472,-0.044642,0.039062,0.001215,0.016318,0.015283,-0.028674,0.026560,0.044529,-0.025930


In [3]:
y

0      151.0
1       75.0
2      141.0
3      206.0
4      135.0
       ...  
437    178.0
438    104.0
439    132.0
440    220.0
441     57.0
Name: target, Length: 442, dtype: float64

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
lin = LinearRegression().fit(X_tr, y_tr)
print('단일 분할 R2:', round(r2_score(y_val, lin.predict(X_val)), 4))   # 약 0.33

단일 분할 R2: 0.3322


In [5]:
from sklearn.model_selection import cross_val_score
scores = cross_val_score(LinearRegression(), X, y, cv=5, scoring='r2')
print('5개 점수:', scores.round(3))
print('평균 R2 :', round(scores.mean(), 4))
# 실측: 5개 점수 [0.43  0.523 0.483 0.426 0.55 ] / 평균 0.4823

5개 점수: [0.43  0.523 0.483 0.426 0.55 ]
평균 R2 : 0.4823


In [6]:
from sklearn.linear_model import Ridge, Lasso
for name, model in [('Ridge', Ridge(alpha=1)), ('Lasso', Lasso(alpha=0.1))]:
    s = cross_val_score(model, X, y, cv=5, scoring='r2').mean()
    print(f'{name} 평균 R2: {s:.4f}')
# 실측: Ridge 평균 R2 0.4102 / Lasso 평균 R2 0.4795

Ridge 평균 R2: 0.4102
Lasso 평균 R2: 0.4795


In [7]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(Ridge(), {'alpha': [0.01, 0.1, 1, 10, 100]}, cv=5, scoring='r2')
grid.fit(X_tr, y_tr)
print('최적 alpha :', grid.best_params_)          # {'alpha': 0.01}
print('최적 CV R2 :', round(grid.best_score_, 4))  # 0.5267
print('검증 R2    :', round(r2_score(y_val, grid.predict(X_val)), 4))  # 0.33

최적 alpha : {'alpha': 0.01}
최적 CV R2 : 0.5267
검증 R2    : 0.33


In [8]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
pipe = make_pipeline(StandardScaler(), Ridge(alpha=1))
s = cross_val_score(pipe, X, y, cv=5, scoring='r2').mean()
print('파이프라인 평균 R2:', round(s, 4))   # 0.4822
# 스케일링이 각 fold의 train에만 fit → 누수 없는 안전한 교차검증
# 값은 4)Ridge와 비슷하지만 핵심은 '누수 없이 얻은 정직한 점수'

파이프라인 평균 R2: 0.4822
